# Claims Validation Notebook

This notebook validates and inspects claims data from the warehouse database.

## Setup


In [ ]:
# Install package first: pip install -e .
import pandas as pd
from src.sdk import ClaimsAnalyst

# Use SDK to access data - do NOT access database directly
analyst = ClaimsAnalyst(db_path="../warehouse.db")

print("ClaimsAnalyst SDK initialized")
print("Use SDK methods to access data from database")


## Load Claims Data

Load claims using SDK - analysts should NOT access database directly.


In [ ]:
# Load claims from database using SDK
claims = analyst.load_from_database(layer="silver")

# Convert to DataFrame for analysis
df = analyst.to_dataframe(claims)

print(f"Loaded {len(claims)} claims from database")
print(f"\nDataFrame shape: {df.shape}")
df.head()


## Get Summary Statistics

Use SDK method to get pre-calculated statistics.


In [ ]:
# Get summary statistics - SDK does all calculations
stats = analyst.get_summary_statistics(claims)

print("Summary Statistics:")
print(f"  Total Claims: {stats['total_claims']}")
print(f"  Total Amount: ${stats['total_amount']:,.2f}")
print(f"  Average Amount: ${stats['average_amount']:,.2f}")
print(f"\nBy Status:")
for status, count in stats['by_status'].items():
    print(f"  {status}: {count}")
print(f"\nBy Type:")
for claim_type, count in stats['by_type'].items():
    print(f"  {claim_type}: {count}")


## Data Quality Checks

Simple pandas operations on DataFrame (no business logic).


In [ ]:
# Simple data quality checks using pandas (no business logic)
print("Null values:")
print(df.isnull().sum())

print("\nClaim amount statistics:")
print(df['claim_amount'].describe())

print("\nStatus distribution:")
print(df['status'].value_counts())


## Aggregate Claims by Policy

Use SDK method for aggregation - business logic is in SDK.


In [ ]:
# Aggregate by policy using SDK
claims_by_policy = analyst.aggregate_by_policy(claims)

print(f"Claims grouped by {len(claims_by_policy)} policies\n")

# Show summary for each policy
for policy_id, policy_claims in list(claims_by_policy.items())[:5]:
    total = analyst.get_total_claims(policy_claims)
    print(f"Policy {policy_id}:")
    print(f"  Claims: {len(policy_claims)}")
    print(f"  Total: ${total:,.2f}")
    print()


## High-Value Claims

Use SDK filter method.


In [ ]:
# Get high-value claims using SDK
from decimal import Decimal

high_value = analyst.get_high_value_claims(claims, threshold=Decimal("50000.00"))
print(f"High-value claims (>$50k): {len(high_value)}")

if high_value:
    # Convert to DataFrame for display
    high_value_df = analyst.to_dataframe(high_value)
    high_value_df[['claim_id', 'policy_id', 'member_id', 'claim_amount', 'status', 'claim_type']]


## DataFrame Analysis

Simple pandas operations for visualization (no business logic).


In [ ]:
# Simple pandas operations for visualization
print("Claims by Policy (pandas groupby):")
policy_summary = df.groupby('policy_id')['claim_amount'].agg(['sum', 'mean', 'count']).round(2)
policy_summary.columns = ['total', 'average', 'count']
policy_summary.sort_values('total', ascending=False)


## Notes

- All data access is through SDK methods
- No direct database access
- Business logic is in SDK, not notebooks
- Use pandas only for simple visualization

In [ ]:
print("Analysis complete")
print("\nRemember: Always use SDK methods, never access database directly!")
